In [8]:
import pandas as pd
import numpy as np
import yfinance as yf
import scipy.optimize
import matplotlib.pyplot as plt


RISK_FREE_RATE = 0.05  # Risk-free rate for Sharpe ratio calculation

#Tickers
TICKERS = ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'TSLA', 'META', 'NVDA', 'JPM', 'V', 'JNJ']

#download SPY ETF data for comparison from aug 2025 to aug 2026
SPY = yf.download('SPY', start='2025-08-01', end='2026-08-01')['Close']

[*********************100%***********************]  1 of 1 completed


In [9]:
# Download historical daily adjusted close prices for the specified tickers
prices = {}
daily_returns = {}
for ticker in TICKERS:
    data = yf.download(ticker, start='2025-08-01', end='2026-08-01')['Close']
    prices[ticker] = data
    daily_returns[ticker] = data.pct_change().dropna()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [11]:
returns_df = pd.DataFrame({
    ticker: data.squeeze("columns")
    for ticker, data in daily_returns.items()
})

# SPY daily returns
spy_prices = SPY.squeeze("columns")
spy_returns = spy_prices.pct_change().dropna()
spy_returns = spy_returns.reindex(returns_df.index).dropna()

# return, variance, and volatility per stock + SPY
annual_return = returns_df.mean() * 252
annual_variance = returns_df.var() * 252
annual_volatility = np.sqrt(annual_variance)

stats_df = pd.DataFrame({
    'Annual Return': annual_return,
    'Annual Variance': annual_variance,
    'Annual Volatility': annual_volatility,
})

spy_stats = pd.Series({
    'Annual Return': spy_returns.mean() * 252,
    'Annual Variance': spy_returns.var() * 252,
    'Annual Volatility': np.sqrt(spy_returns.var() * 252),
}, name='SPY')

stats_df = pd.concat([stats_df, spy_stats.to_frame().T])

print("Annualized Return / Variance / Volatility:")
print(stats_df)

Annualized Return / Variance / Volatility:
       Annual Return  Annual Variance  Annual Volatility
AAPL        0.463989         0.066845           0.258543
GOOGL       0.692269         0.103021           0.320968
MSFT       -0.063753         0.100845           0.317562
AMZN        0.294055         0.117510           0.342797
TSLA        0.136194         0.215117           0.463807
META       -0.224120         0.145633           0.381619
NVDA        0.213295         0.132977           0.364660
JPM         0.241210         0.050307           0.224292
V           0.108436         0.048416           0.220036
JNJ         0.472056         0.033785           0.183808
SPY         0.204460         0.016440           0.128217


In [12]:
#creating covariance matrix
returns_cov_matrix = returns_df.cov() * 252  # Annualize

print("Covariance Matrix:")
print(returns_cov_matrix)

Covariance Matrix:
           AAPL     GOOGL      MSFT      AMZN      TSLA      META      NVDA  \
AAPL   0.066845  0.016327  0.009240  0.010530  0.025524  0.017755  0.011837   
GOOGL  0.016327  0.103021  0.012276  0.053102  0.060897  0.039263  0.031960   
MSFT   0.009240  0.012276  0.100845  0.038766  0.032010  0.018007  0.033520   
AMZN   0.010530  0.053102  0.038766  0.117510  0.057230  0.055323  0.037962   
TSLA   0.025524  0.060897  0.032010  0.057230  0.215117  0.062671  0.069430   
META   0.017755  0.039263  0.018007  0.055323  0.062671  0.145633  0.047156   
NVDA   0.011837  0.031960  0.033520  0.037962  0.069430  0.047156  0.132977   
JPM    0.009029  0.014414  0.006788  0.013263  0.017993  0.019089  0.015661   
V      0.012790  0.005014  0.012477  0.013396 -0.001309  0.015318 -0.008275   
JNJ    0.001909 -0.000022 -0.014271 -0.009683 -0.014246 -0.008867 -0.014401   

            JPM         V       JNJ  
AAPL   0.009029  0.012790  0.001909  
GOOGL  0.014414  0.005014 -0.000022

In [13]:
#helper function to calculate portfolio return
def port_return(w, mu):
    return np.dot(w, mu)

#helper function to calculate portfolio volatility
def port_volatility(w, cov_matrix):
    return np.sqrt(np.dot(w.T, np.dot(cov_matrix, w)))



In [ ]:
#long-short frontier closed form

#moving inputs into numpy arrays for calculations
mu = annual_return.values
cov_matrix = returns_cov_matrix.values

Sigma_inv = np.linalg.inv(cov_matrix)

#ones vector
ones = np.ones(len(mu))

A = np.dot(ones.T, np.dot(Sigma_inv, ones))
B = np.dot(mu.T, np.dot(Sigma_inv, ones))
C = np.dot(mu.T, np.dot(Sigma_inv, mu))
D = A * C - B ** 2

#return grid; starting near min ad stepping up by 2% till highest stock's return
ret_grid = np.arange(annual_return.min(), annual_return.max(), 0.02)

#loop thru grid and calculate weights, vol, sharpe 
volatilities = []
returns = []
weights = []
for ret in ret_grid:
    # Calculate the weights for the given return
    w = Sigma_inv @ ((C * ret - B)/D * mu + (A - B*ret)/D * ones)

    # Calculate the volatility for the given weights
    vol = port_volatility(w, cov_matrix)

    # store resulting weights and metrics with a dataframe with columns
    volatilities.append(vol)
    returns.append(ret)
    weights.append(w)

ls_frontier_df = pd.DataFrame({
    'Return': returns,
    'Volatility': volatilities,
    'Weights': weights
})  
